# Phase 6d — End-to-End SQuAD Generation Evaluation (Chunk 3)

This final Phase 6 notebook compares Dense Baseline, Static Strong, and Self-Healing RAG on answer quality, controller abstention, safe refusal, lexical support, local NLI groundedness, recovery-conditional quality, and compute latency. The detector is loaded from the frozen TRAIN-only Chunk 2 artifact and is never recalibrated here. No paid API or service is used.

## 1. Colab T4 environment

Select **Runtime → Change runtime type → T4 GPU**, set `REPOSITORY_URL`, and run all cells. This notebook intentionally refuses CPU model execution.

In [ ]:
from pathlib import Path
import json
import os
import random
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/YOUR_GITHUB_USERNAME/adaptive-rag.git'
PROJECT_DIR = Path('/content/adaptive-rag')
if not (PROJECT_DIR / 'src').exists():
    if 'YOUR_GITHUB_USERNAME' in REPOSITORY_URL:
        raise RuntimeError('Set REPOSITORY_URL, then run this cell again.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Working directory:', Path.cwd())

In [ ]:
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    raise RuntimeError('Chunk 3 model evaluation must run on a Colab GPU, not CPU.')
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0))

## 2. Configuration and frozen Chunk 2 inputs

Smoke mode targets 42 observations before availability adjustments: 10 clean, 6 hard, 6 paraphrased, 10 controlled-missing, and 10 natural-unanswerable. Full mode targets 400: 150, 50, 50, 75, and 75. If Chunk 2 selected fewer hard cases, every available fixed hard case is retained and the actual count is reported.

In [ ]:
SMOKE_TEST = True
FORCE_RERUN = True
RESULTS_DIR = Path('results')
REQUIRED_CHUNK2 = {
    'calibration': RESULTS_DIR / 'squad_detector_calibration.json',
    'stress_manifest': RESULTS_DIR / 'squad_stress_manifest.json',
    'stress_per_example': RESULTS_DIR / 'squad_stress_per_example.csv',
    'stress_metrics': RESULTS_DIR / 'squad_stress_metrics.json',
}
OUTPUTS = {
    'manifest': RESULTS_DIR / 'squad_generation_manifest.json',
    'per_example': RESULTS_DIR / 'squad_generation_per_example.csv',
    'metrics': RESULTS_DIR / 'squad_generation_metrics.json',
    'summary': RESULTS_DIR / 'squad_generation_summary.md',
    'generation_cache': RESULTS_DIR / 'squad_generation_cache.jsonl',
    'nli_claims': RESULTS_DIR / 'squad_nli_per_claim.csv',
}
print({'smoke_test': SMOKE_TEST, 'force_rerun': FORCE_RERUN})

## 3. Load frozen artifacts and build the fixed generation subset

The loader requires all four Chunk 2 artifacts and reads `selected_config` from `squad_detector_calibration.json`. Selection uses only frozen IDs/tracks and never uses validation scores or outcomes.

In [ ]:
from src.evaluation import (
    Chunk2ArtifactPaths,
    GenerationSubsetConfig,
    SquadStressSamplingConfig,
    load_chunk2_artifacts,
    prepare_generation_subset,
    prepare_squad_stress_dataset,
)

artifact_paths = Chunk2ArtifactPaths(**REQUIRED_CHUNK2)
chunk2 = load_chunk2_artifacts(artifact_paths)
sampling_values = dict(chunk2.stress_manifest['sampling_config'])
chunk2_sampling = SquadStressSamplingConfig(**sampling_values)
stress_data = prepare_squad_stress_dataset(chunk2_sampling, REQUIRED_CHUNK2['stress_manifest'])
subset_config = GenerationSubsetConfig(
    clean_answerable_count=10 if SMOKE_TEST else 150,
    hard_distractor_count=6 if SMOKE_TEST else 50,
    paraphrase_count=6 if SMOKE_TEST else 50,
    controlled_missing_count=10 if SMOKE_TEST else 75,
    natural_unanswerable_count=10 if SMOKE_TEST else 75,
    seed=SEED,
    cache_dir='.cache/huggingface',
)
generation_data = prepare_generation_subset(
    chunk2, stress_data.documents, subset_config, OUTPUTS['manifest']
)
REUSE_SAVED = False
if not FORCE_RERUN and OUTPUTS['per_example'].exists() and OUTPUTS['metrics'].exists():
    saved_generation = json.loads(OUTPUTS['metrics'].read_text(encoding='utf-8'))
    REUSE_SAVED = (
        saved_generation.get('dataset', {}).get('fingerprint') == generation_data.fingerprint
        and saved_generation.get('chunk2_artifact_sha256') == chunk2.sha256
    )
    if not REUSE_SAVED:
        print('Saved generation results do not match this subset/artifact set; rerunning.')
print('Frozen detector label:', chunk2.calibration['label'])
print('Generation observations:', len(generation_data.examples))
print('Examples per category:', json.dumps(generation_data.category_counts, indent=2))
print('Generation manifest:', generation_data.manifest_path)
print('Reuse completed results:', REUSE_SAVED)

## 4. Initialize free local models

Generation uses deterministic `Qwen/Qwen2.5-1.5B-Instruct`. NLI uses configurable `cross-encoder/nli-MiniLM2-L6-H768`, evaluates meaningful claims against individual top passages, truncates each pair to 384 tokens, and batches on the T4.

In [ ]:
from src.evaluation import LocalNLIGroundednessEvaluator, NLIConfig
from src.rag import BM25Retriever, CrossEncoderReranker, FAISSRetriever, LocalQwenGenerator, RAGConfig

dense = bm25 = reranker = generator = nli_evaluator = None
if not REUSE_SAVED:
    rag_config = RAGConfig(
        embedding_model_name='sentence-transformers/all-MiniLM-L6-v2',
        generation_model_name='Qwen/Qwen2.5-1.5B-Instruct',
        embedding_batch_size=128,
        max_new_tokens=64,
        temperature=0.0,
        do_sample=False,
    )
    index_dir = Path('.cache/squad_stress_faiss') / stress_data.fingerprint
    dense = FAISSRetriever(rag_config.embedding_model_name, device='cuda', batch_size=128)
    if (index_dir / 'documents.faiss').exists():
        dense.load(index_dir)
    else:
        dense.build(stress_data.documents)
        dense.save(index_dir)
    bm25 = BM25Retriever(stress_data.documents)
    reranker = CrossEncoderReranker(device='cuda', batch_size=64)
    generator = LocalQwenGenerator(rag_config)
    nli_config = NLIConfig(
        model_name='cross-encoder/nli-MiniLM2-L6-H768',
        batch_size=64, max_length=384, max_passages=3,
        entailment_threshold=0.70, contradiction_threshold=0.70, device='cuda',
    )
    nli_evaluator = LocalNLIGroundednessEvaluator(nli_config)
    print('Dense device:', dense.device)
    print('Reranker device:', reranker.device)
    print('Qwen device:', generator.device)
    print('NLI device/model:', nli_evaluator.device, nli_config.model_name)

## 5. Run exactly three systems

Dense Baseline uses FAISS only. Static Strong uses dense + BM25 + RRF + cross-encoder with no diagnostics or retries. Self-Healing uses the existing LangGraph retrieval/recovery route and the frozen TRAIN-calibrated detector; Qwen is invoked only when the controller does not abstain. Generation cache keys omit the system name and include the full semantic prompt inputs, allowing safe cross-system reuse.

In [ ]:
from src.evaluation import (
    SquadGenerationEvaluator,
    load_generation_records,
    representative_error_cases,
)

if REUSE_SAVED:
    records = load_generation_records(OUTPUTS['per_example'])
    payload = json.loads(OUTPUTS['metrics'].read_text(encoding='utf-8'))
else:
    evaluator = SquadGenerationEvaluator(
        dense, bm25, reranker, generator, chunk2.frozen_detector_config,
        generation_cache_path=OUTPUTS['generation_cache'],
        nli_evaluator=nli_evaluator,
        dense_top_k=5, static_candidate_k=20, final_top_k=5,
        generation_top_k=3, output_dir=RESULTS_DIR,
    )
    records, payload = evaluator.run(
        generation_data, chunk2, nli_config=nli_config,
        progress_callback=lambda i, n, example, system: print(
            f'{i}/{n} {example.track} {system}'
        ) if i % 10 == 0 or i == n else None,
    )
print('Per-system rows:', len(records))

## 6. Required final summary

In [ ]:
metrics = payload['metrics']
print('=== DATASET ===')
print(json.dumps(payload['dataset'], indent=2))

print('\n=== ANSWERABLE QUALITY / GROUNDEDNESS / PER-TRACK ===')
for system, result in metrics['by_system'].items():
    overall = result['overall']
    print(f'\n{system}', {
        'Recall@5': overall['recall_at_5'],
        'EM': overall['answer_em'],
        'F1': overall['answer_f1'],
        'EM | gold@5': overall['answer_em_given_gold_retrieved_at_5'],
        'F1 | gold@5': overall['answer_f1_given_gold_retrieved_at_5'],
        'LEXICAL HEURISTIC coverage': overall['lexical_heuristic_mean_coverage'],
        'NLI entailed': overall['nli_entailed_percentage'],
        'NLI neutral': overall['nli_neutral_percentage'],
        'NLI contradicted': overall['nli_contradicted_percentage'],
        'NLI fully grounded': overall['nli_fully_grounded_rate'],
    })
    for track, track_metrics in result['per_track'].items():
        print(' ', track, track_metrics)

print('\n=== CONTROLLED MISSING EVIDENCE ===')
print('Controller abstention:', json.dumps(metrics['controller_abstention'], indent=2))
for system, result in metrics['by_system'].items():
    missing = result['per_track'].get('CONTROLLED_MISSING_EVIDENCE', {})
    natural = result['per_track'].get('NATURAL_UNANSWERABLE', {})
    print(system, {
        'controlled_safe_refusal_rate': missing.get('safe_refusal_rate'),
        'controlled_unsupported_answer_rate': missing.get('unsupported_answer_rate'),
        'natural_safe_refusal_rate_descriptive': natural.get('safe_refusal_rate'),
        'natural_substantive_answer_rate_descriptive': (
            None if not natural else 1.0 - natural.get('safe_refusal_rate', 0.0)
        ),
    })

print('\n=== SELF-HEALING ===')
print(json.dumps(metrics['self_healing_recovery'], indent=2))
print('Latency overhead vs Static Strong:', metrics['self_healing_latency_overhead_vs_static_strong'])

print('\n=== FINAL THREE-WAY COMPARISON ===')
comparison = metrics['three_way_comparison']
columns = ['recall_at_5','answer_em','answer_f1','nli_fully_grounded_rate','safe_refusal_rate','unsupported_answer_rate','mean_latency_seconds','median_latency_seconds','controller_abstention_rate','recovery_attempt_rate']
print('System'.ljust(22), *[name[:16].rjust(17) for name in columns])
for system, values in comparison.items():
    formatted = [('N/A' if values[name] is None else f'{values[name]:.4f}').rjust(17) for name in columns]
    print(system.ljust(22), *formatted)

print('\nDetector calibrated only on TRAIN:', payload['scientific_integrity']['detector_calibration_split'])
print('Validation use:', payload['scientific_integrity']['validation_usage'])
print('Paid services used:', payload['scientific_integrity']['paid_services_used'])

## 7. Representative error analysis

Examples are selected after evaluation only for inspection; none are discarded from metrics. Context is represented by IDs and the answer is truncated for readability.

In [ ]:
cases = representative_error_cases(records, limit=2)
for category, examples in cases.items():
    print(f'\n{category}')
    if not examples:
        print('  none observed')
    for example in examples:
        print(json.dumps(example, indent=2, ensure_ascii=False))

## 8. Outputs and interpretation

Lexical groundedness is labeled **LEXICAL HEURISTIC** and is not semantic faithfulness. NLI entailment, neutral, contradiction, and fully-grounded rates remain separate. `total_latency_seconds` uses measured uncached generation compute stored in the cache so identical cross-system prompts are compared fairly; `observed_total_latency_seconds` shows actual resumed/cache-hit wall time. Full T4 runtime is expected to be roughly **2–4 hours** on a cold cache; smoke mode roughly **20–40 minutes**. Runtime varies with model downloads, cache state, generated answer length, and recovery frequency.

In [ ]:
print('Artifacts:')
for name, path in OUTPUTS.items():
    print(f'- {name}: {path} (exists={path.exists()})')
print('\nPhase 6 complete after this Chunk 3 generation evaluation. No deployment or Chunk 4 work was added.')